# nano Beatrix v1.1 — the gate screen

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_1_gate.ipynb)

The note arms showed that a body writes notes of its own only behind the carry habit, and that the notes it writes come from a fixed
extractor. The gate removes the extractor: the body proposes a note wherever its reading-with-notes protocol opens a frame, and its
own loss on the text that follows decides whether the note is kept. A note is KEPT at a horizon H and a margin m when the next H
bytes cost at least m bits less with the note than without it, and at least m bits less than with the note's value replaced by
spaces (the key and the frame kept). Both controls are Toolformer's; the horizon and the margin are scanned here, never set. A
random note (a one-record frame with a key and a value from the bed's vocabulary) is inserted at the same positions and scored the
same way: the control must fail.

Two row sets a candidate: the note gauge's rows (held-out documents holding a fact, the question far at the end; the notes read
from byte 128) and the carry gauge's plain rows (no fact selection, no question). The horizon never crosses the question.

| mark | rule |
|---|---|
| PROPOSES | a candidate writes 1.0 note a row or more on the fact rows (below it there is nothing to gate: the candidate is dropped) |
| THE GATE CHOOSES | at (256, 1) the kept share of proposed notes lies between 0.05 and 0.80 |
| THE GAIN IS ABOUT THE TEXT | the kept notes' median gain over no note is 2 bits or more at (256, 1), and the gains with the value withheld sit below the gains without the note |
| REAL FACTS | 0.50 or more of the kept notes at (256, 1) carry a value found verbatim in the preceding 1,024 bytes |
| THE CONTROL FAILS | the random note's kept share at (256, 1) is 0.05 or less |
| COST | the wall time a final a mode on this card |

Nothing trains here; every read is forward-only on stored finals. The records ship to the Hub as JSON under the first candidate's
prefix. The record of the screen and its reading live in the research notes; the library holds the code (`gate_screen.py`,
`nano_v1.gate_stored`).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after this cell on a fresh runtime (the install replaces numpy
# under a kernel that has imported it), then run this cell again and the rest.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.54"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 54), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the finals are read from the repo and the records ship to it (Secrets: the key icon, notebook access on)'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_v1_1_gate'                       # the session folder: the held-out head, the bed's folders, the records
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: the finals read, the records written
NAMES = ['nano-pollux-dual-v1.1-cf-nt',               # the one body that kept the note habit (2.01 notes a row with the note absent)
         'nano-pollux-dual-v1.1-mx',                  # the mixed arm (carries 4.36 a row on carry rows; plain rows expected 0.00)
         'nano-pollux-dual-v1.1-nt',                  # the single-phase note arm (0.00 notes a row: the premise read says so)
         'nano-pollux-dual-v1.1-ct']                  # the control (0.00 frames a row on plain text)
MODES = ('self', 'plain')                             # the note gauge's fact rows; the carry gauge's plain rows
ROWS = 192                                            # the gauges' own 192 rows (file order), the same rows every read uses
HORIZONS = '64,128,256,512'                           # bytes after the note that are scored
MARGINS = '0.25,0.5,1,2,4'                            # bits the note must save against both controls
MB, GA = 16, 4                                        # the 2s shape: the steps-per-byte constant the final step is derived from
TPS = MB * GA * V1.CTX
print('decision block:', len(NAMES), 'candidates:', ', '.join(NAMES), '| modes', MODES, '| rows', ROWS, '| horizons', HORIZONS,
      '| margins', MARGINS, '| reads from and ships to', REPO)


In [ ]:
# C2 (P) — PREFLIGHT: every candidate's final in the repo, the held-out head (the only data the screen needs), the push probe, and a
# 4-row smoke of the screen on the first candidate (about a minute). Nothing trains here.
import json, time
from huggingface_hub import HfApi
from geolip.nano_beatrix.bed_corpus import write_heldout
api = HfApi(token=HF_TOKEN)
files = set(api.list_repo_files(REPO))
missing = []
for n in NAMES:
    step = V1.plan_steps(n, TPS)
    path = f'{n}/checkpoints/step_{step:08d}.safetensors'
    if path not in files:
        missing.append(path)
assert not missing, f'not every candidate has its final in the repo: missing {sorted(set(missing))}'
print('finals: every candidate has its final;', ', '.join(f'{n} at {V1.plan_steps(n, TPS):,}' for n in NAMES))
os.makedirs(f'{OUT}/bed', exist_ok=True)
held = write_heldout(f'{OUT}/bed')
import shutil; os.makedirs(f'{OUT}/smoke/bed', exist_ok=True); shutil.copy(held, f'{OUT}/smoke/bed/')   # the smoke's own folder gets the same held-out (0.1.54)
print('held-out head:', held)

PUSH_OK = False
path = 'preflight/push_probe_v1_1_gate.json'          # its own probe file
body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                   'candidates': NAMES}).encode()
api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano v1.1 gate screen: push probe')
PUSH_OK = path in set(api.list_repo_files(REPO))
print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)

# the smoke: the screen's child on 4 fact rows of the first candidate, its record read back (the real reads in C3 use their own records)
n0 = NAMES[0]
t0 = time.time()
smoke = V1.gate_stored(n0, V1.plan_steps(n0, TPS), f'{OUT}/smoke', repo=REPO, mode='self', token=HF_TOKEN, rows=4,
                       extra=dict(horizons=HORIZONS, margins=MARGINS, dump=1))
assert smoke['rows'] == 4 and 'table' in smoke and f'256|1.0' in smoke['table'], sorted(smoke)
print(f"smoke on {n0}: {smoke['notes_a_row']:.2f} notes a row on 4 rows; kept at (256, 1) {smoke['table']['256|1.0']['kept']} of "
      f"{smoke['table']['256|1.0']['scored']} scored; {time.time() - t0:.0f} s on this card")


In [ ]:
# C3 (R) — THE PREMISE READ AND THE SCREEN: every candidate, both modes; the record of each read is written once under OUT/bed and
# returned as it is on a rerun. A candidate that proposes under 1.0 note a row on the fact rows fails PROPOSES and is still read on
# the plain rows (the premise read is the result for it). Minutes a read on a large card.
assert PUSH_OK, 'the push probe did not read back: the records would not ship'
import time
REC = {}
t_all = time.time()
for n in NAMES:
    step = V1.plan_steps(n, TPS)
    for mode in MODES:
        t0 = time.time()
        r = V1.gate_stored(n, step, OUT, repo=REPO, mode=mode, token=HF_TOKEN, rows=ROWS, extra=dict(horizons=HORIZONS, margins=MARGINS, dump=1))
        REC[(n, mode)] = r
        t = r['table'].get('256|1.0', {})
        print(f"{n} {mode}: {r['notes_a_row']:.2f} notes a row ({r['rows_with_notes']:.2f} of rows); kept at (256, 1) {t.get('kept_a_row', 0):.2f} a row, "
              f"share {t.get('kept_share')}; control share {t.get('control_kept_share')}; {r['seconds']:.0f} s "
              f"({time.time() - t_all:.0f} s so far)")


In [ ]:
# C4 — THE TABLES, THE CURVES AND THE MARKS. Run any time after C3.
from IPython.display import Markdown, display

def f(x, nd=3):
    return '-' if x is None else f'{x:.{nd}f}'

def at(r, H=256, m=1.0):
    return r['table'].get(f'{H}|{float(m)}', {})

# the premise: notes a row by candidate and mode, and the reference setting
rows = ['| candidate | mode | notes a row | rows with a note | parsed | kept a row at (256, 1) | kept share | control kept share | median gain without, kept | median gain withheld, kept | kept: value in the text before | kept: key the extractor finds | seconds |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|---|']
for (n, mode), r in REC.items():
    t = at(r)
    gw, gh = r.get('gain_without_kept') or {}, r.get('gain_withheld_kept') or {}
    rows.append(f"| {n} | {mode} | {r['notes_a_row']:.2f} | {r['rows_with_notes']:.2f} | {f(r['parsed_share'], 2)} | {f(t.get('kept_a_row'), 2)} | {f(t.get('kept_share'), 2)} | "
                f"{f(t.get('control_kept_share'), 3)} | {f(gw.get('median'), 1)} | {f(gh.get('median'), 1)} | {f(r.get('kept_value_in_text_before'), 2)} | "
                f"{f(r.get('kept_key_found_by_extractor'), 2)} | {r['seconds']:.0f} |")
display(Markdown('\n'.join(rows)))

# the curves: the kept share over the margin at H = 256 and over the horizon at m = 1
Hs = [int(x) for x in HORIZONS.split(',')]
Ms = [float(x) for x in MARGINS.split(',')]
cur = ['| candidate | mode | ' + ' | '.join(f'm={m} at H=256' for m in Ms) + ' | ' + ' | '.join(f'H={H} at m=1' for H in Hs) + ' |',
       '|---|---|' + '---|' * (len(Ms) + len(Hs))]
for (n, mode), r in REC.items():
    cur.append(f'| {n} | {mode} | ' + ' | '.join(f(at(r, 256, m).get('kept_share'), 2) for m in Ms) + ' | '
               + ' | '.join(f(at(r, H, 1.0).get('kept_share'), 2) for H in Hs) + ' |')
display(Markdown('\n'.join(cur)))

# the marks, by the rule written before the run (the fact rows decide; the plain rows are read beside)
marks = ['| candidate | PROPOSES | THE GATE CHOOSES | THE GAIN IS ABOUT THE TEXT | REAL FACTS | THE CONTROL FAILS | pass |', '|---|---|---|---|---|---|---|']
ok = lambda c: 'MET' if c else 'NOT MET'
for n in NAMES:
    r = REC.get((n, 'self'))
    if r is None:
        continue
    t = at(r)
    proposes = r['notes_a_row'] >= 1.0
    share = t.get('kept_share')
    chooses = share is not None and 0.05 <= share <= 0.80
    gw, gh = r.get('gain_without_kept') or {}, r.get('gain_withheld_kept') or {}
    about_text = (gw.get('median') or 0) >= 2.0 and gh.get('median') is not None and gh['median'] <= gw['median']
    real = (r.get('kept_value_in_text_before') or 0) >= 0.50
    ctrl = t.get('control_kept_share') is not None and t['control_kept_share'] <= 0.05
    marks.append(f"| {n} | {ok(proposes)} ({r['notes_a_row']:.2f}) | {ok(chooses)} ({f(share, 2)}) | {ok(about_text)} ({f(gw.get('median'), 1)} / {f(gh.get('median'), 1)}) | "
                 f"{ok(real)} ({f(r.get('kept_value_in_text_before'), 2)}) | {ok(ctrl)} ({f(t.get('control_kept_share'), 3)}) | "
                 f"{'PASS' if proposes and chooses and about_text and real and ctrl else ('NOT READY' if proposes else 'NO PROPOSALS')} |")
display(Markdown('\n'.join(marks)))

# a look at the notes kept and dropped at (256, 1) on the first candidate that proposes
for n in NAMES:
    r = REC.get((n, 'self'))
    if r and r['notes_a_row'] >= 1.0:
        print('kept, ' + n + ':')
        for e in r['kept_examples'][:6]:
            print(f"  row {e['row']} @ {e['pos']}: key {e['key']!r} value {e['value']!r}; gain without {e['gain_without']:.1f}, withheld {e['gain_withheld']:.1f}; in the text before: {e['value_in_text_before']}")
        print('dropped, ' + n + ':')
        for e in r['dropped_examples'][:6]:
            print(f"  row {e['row']} @ {e['pos']}: key {e['key']!r} value {e['value']!r}; gain without {e['gain_without']:.1f}, withheld {e['gain_withheld']:.1f}; in the text before: {e['value_in_text_before']}")
        break


In [ ]:
# C5 — SHIP THE RECORDS: the reads of C3 as one JSON file under the first candidate's prefix (<craft>/reads/gate_<stamp>.json),
# so the record is read from the Hub like every other result. Run after C3; it trains nothing.
import json, time
from huggingface_hub import HfApi
stamp = time.strftime('%Y%m%d_%H%M')
rec = dict(stamp=stamp, names=NAMES, modes=list(MODES), rows=ROWS, horizons=HORIZONS, margins=MARGINS,
           reads={n + '|' + mode: {k: v for k, v in r.items() if k != 'dump'} for (n, mode), r in REC.items()})
p = f'{OUT}/gate_{stamp}.json'
json.dump(rec, open(p, 'w', encoding='utf-8'), indent=1)
dest = f'{NAMES[0]}/reads/gate_{stamp}.json'
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=p, path_in_repo=dest, repo_id=REPO,
                                  commit_message='the gate screen: the premise read and the screen on the candidates (notebook C3)')
print('shipped', dest)


### Changing the run

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. A read already on disk under OUT/bed is returned as it is.

| to | the edit in C1 |
|---|---|
| another body's candidates | `NAMES = ['nano-castor-v1.1-cf-nt', 'nano-castor-v1.1-mx', 'nano-castor-v1.1-ct']` |
| the plain rows only | `MODES = ('plain',)` |
| a finer margin scan | `MARGINS = '0.1,0.25,0.5,1,2,4,8'` |
| fewer rows for a quick look | `ROWS = 48` (the record then is not the 192-row read) |

The session folder is the Colab disk; the repo holds the records.
